In [ ]:
# cv practical assignment 4

In [ ]:
# question 1

In [3]:
import cv2

# Initialize video capture (0 for webcam, or pass a video file path e.g. "corridor.mp4")
video_path = 0 
cap = cv2.VideoCapture("person_walking_in_bedroom.mp4")

if not cap.isOpened():
    print("Error: Could not open video source.")
    exit()

# Read the first frame
ret, frame1 = cap.read()
ret, frame2 = cap.read()

# Minimum contour area to trigger motion detection (filters out small noise)
MIN_CONTOUR_AREA = 1000

while cap.isOpened() and ret:
    # 1. Compute absolute difference between consecutive frames
    diff = cv2.absdiff(frame1, frame2)

    # 2. Convert to grayscale
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)

    # 3. Apply Gaussian Blur to smooth out high-frequency noise
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # 4. Threshold to get a binary image (motion = white, background = black)
    _, thresh = cv2.threshold(blur, 20, 255, cv2.THRESH_BINARY)

    # 5. Dilate thresholded image to fill in gaps inside contours
    dilated = cv2.dilate(thresh, None, iterations=3)

    # 6. Find contours of the motion regions
    contours, _ = cv2.findContours(dilated, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    motion_detected = False

    for contour in contours:
        # Ignore small movements (e.g., small camera jitter or shadows)
        if cv2.contourArea(contour) < MIN_CONTOUR_AREA:
            continue

        motion_detected = True

        # Get bounding box coordinates for the moving region
        (x, y, w, h) = cv2.boundingRect(contour)

        # Draw bounding box around detected motion
        cv2.rectangle(frame1, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Display status on the video feed
    if motion_detected:
        cv2.putText(
            frame1,
            "Status: Motion Detected",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255),
            2,
        )
    else:
        cv2.putText(
            frame1,
            "Status: Clear",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2,
        )

    # Display frames
    cv2.imshow("Corridor CCTV Feed", frame1)
    cv2.imshow("Motion Threshold", dilated)

    # Update frames for the next iteration
    frame1 = frame2
    ret, frame2 = cap.read()

    # Press 'q' or 'ESC' to exit
    key = cv2.waitKey(30) & 0xFF
    if key == ord("q") or key == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# question 2

In [4]:
import cv2

# Initialize video stream (use 0 for webcam or replace with path to video file, e.g., 'parking.mp4')
video_path = "person_walking_in_bedroom.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open video source.")
    exit()

# Initialize MOG2 Background Subtractor
# history: number of frames used to build the background model
# varThreshold: threshold on the squared Mahalanobis distance (controls sensitivity)
# detectShadows: True marks shadows in gray (127), which helps exclude them from contours
bg_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=500, varThreshold=50, detectShadows=True
)

# Morphological kernel to clean noise and bridge gaps in vehicle silhouettes
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))

# Minimum contour area to trigger vehicle detection (filters out small movement like leaves/birds)
MIN_VEHICLE_AREA = 3000

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # 1. Apply background subtraction
    fg_mask = bg_subtractor.apply(frame)

    # 2. Threshold to remove detected shadows (shadows are marked as 127, motion as 255)
    _, thresh = cv2.threshold(fg_mask, 250, 255, cv2.THRESH_BINARY)

    # 3. Morphological operations: Open to remove small speckles, Close to merge vehicle regions
    clean_mask = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
    clean_mask = cv2.morphologyEx(clean_mask, cv2.MORPH_CLOSE, kernel)

    # 4. Find contours of detected moving regions
    contours, _ = cv2.findContours(
        clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    vehicle_count = 0

    for contour in contours:
        # Ignore motion smaller than typical vehicle threshold
        if cv2.contourArea(contour) < MIN_VEHICLE_AREA:
            continue

        vehicle_count += 1

        # Get bounding box coordinates for moving vehicle
        x, y, w, h = cv2.boundingRect(contour)

        # Draw bounding box and label
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(
            frame,
            "Vehicle",
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            2,
        )

    # Display real-time status overlay
    status_text = f"Moving Vehicles Detected: {vehicle_count}"
    cv2.putText(
        frame,
        status_text,
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 255) if vehicle_count > 0 else (0, 255, 0),
        2,
    )

    # Render video windows
    cv2.imshow("Parking Security Camera", frame)
    cv2.imshow("Foreground Mask (Cleaned)", clean_mask)

    # Press 'q' or 'ESC' to exit
    key = cv2.waitKey(30) & 0xFF
    if key == ord("q") or key == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# question 3

In [1]:
import cv2

# Initialize video source (0 for live webcam, or path to video file e.g., 'surveillance.mp4')
video_path = "person_walking_in_bedroom.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open video feed.")
    exit()

# Initialize MOG2 Subtractor
# history=500: baseline frame history length
# varThreshold=16: variance threshold for pixel classification (lower = higher sensitivity)
# detectShadows=True: distinguishes true motion (255) from cast shadows (127)
bg_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=500, varThreshold=16, detectShadows=True
)

# Structuring element for noise filtering & merging segmented regions
kernel_open = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))

# Minimum contour area (pixels) to qualify as an object
MIN_OBJECT_AREA = 800

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # 1. Obtain foreground mask from MOG2 model
    fg_mask = bg_subtractor.apply(frame)

    # 2. Isolate true motion pixels (255) from shadows (127)
    _, binary_mask = cv2.threshold(fg_mask, 250, 255, cv2.THRESH_BINARY)

    # 3. Morphological Operations
    # OPEN removes small noise dots; CLOSE fills gaps within individual objects
    mask_cleaned = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel_open)
    mask_cleaned = cv2.morphologyEx(mask_cleaned, cv2.MORPH_CLOSE, kernel_close)

    # 4. Find external contours representing independent moving objects
    contours, _ = cv2.findContours(
        mask_cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    detected_objects_count = 0

    for contour in contours:
        # Filter out minor perturbations (leaves, camera shake, small animals)
        if cv2.contourArea(contour) < MIN_OBJECT_AREA:
            continue

        detected_objects_count += 1

        # Calculate bounding box & object centroid
        x, y, w, h = cv2.boundingRect(contour)
        cx, cy = x + w // 2, y + h // 2

        # Draw bounding rectangle around each detected object
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        # Plot centroid point
        cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

        # Label each detected instance with ID and dimensions
        label = f"Obj #{detected_objects_count} [{w}x{h}]"
        cv2.putText(
            frame,
            label,
            (x, y - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            1,
            cv2.LINE_AA,
        )

    # Top overlay bar showing total live detections
    cv2.rectangle(frame, (0, 0), (320, 40), (0, 0, 0), -1)
    status_msg = f"Active Targets: {detected_objects_count}"
    cv2.putText(
        frame,
        status_msg,
        (10, 26),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2,
        cv2.LINE_AA,
    )

    # Display real-time output streams
    cv2.imshow("Multi-Object Detection Feed", frame)
    cv2.imshow("MOG2 Mask (Cleaned)", mask_cleaned)

    # Key exit listener ('q' or 'ESC')
    key = cv2.waitKey(30) & 0xFF
    if key == ord("q") or key == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# question 4

In [2]:
import cv2

# Initialize video capture (0 for live camera, or path to video file e.g., 'surveillance.mp4')
video_path = "person_walking_in_bedroom.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open video source.")
    exit()

# Read the first frame
ret, frame = cap.read()
if not ret:
    print("Error: Failed to read video feed.")
    cap.release()
    exit()

# Initialize OpenCV Object Tracker
# CSRT is highly accurate and handles rotation/occlusion well
# Note: For OpenCV versions < 4.5.0, use cv2.TrackerCSRT_create()
tracker = cv2.TrackerCSRT_create()

# Prompt user to select target object using bounding box
print("=== CONTROLS ===")
print("1. Drag mouse to draw a box around the object to track.")
print("2. Press ENTER or SPACE to confirm selection.")
print("3. Press 'c' to cancel selection.")
print("=================")

# Open ROI (Region of Interest) selector window
bbox = cv2.selectROI("Select Object to Track", frame, fromCenter=False, showCrosshair=True)
cv2.destroyWindow("Select Object to Track")

# Initialize the tracker with the selected bounding box and frame
if bbox[2] > 0 and bbox[3] > 0:  # Check if a valid ROI was selected
    tracker.init(frame, bbox)
else:
    print("No valid object selected. Exiting.")
    cap.release()
    exit()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Update tracker position for the current frame
    success, bbox = tracker.update(frame)

    if success:
        # Tracking success: Draw bounding box
        x, y, w, h = [int(v) for v in bbox]
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        
        # Draw center point
        cx, cy = x + w // 2, y + h // 2
        cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

        # Status text
        cv2.putText(frame, "Tracking Active", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(frame, f"Pos: ({cx}, {cy})", (10, 60), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
    else:
        # Tracking failure: Object lost or out of frame
        cv2.putText(frame, "Tracking Lost!", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    # Display real-time output
    cv2.imshow("Object Tracking Feed", frame)

    # Controls: Press 'r' to re-select object, 'q' or ESC to quit
    key = cv2.waitKey(30) & 0xFF
    if key == ord('q') or key == 27:
        break
    elif key == ord('r'):
        # Re-select object interactively during runtime
        new_bbox = cv2.selectROI("Select Object to Track", frame, fromCenter=False)
        cv2.destroyWindow("Select Object to Track")
        if new_bbox[2] > 0 and new_bbox[3] > 0:
            tracker = cv2.TrackerCSRT_create()
            tracker.init(frame, new_bbox)

cap.release()
cv2.destroyAllWindows()

AttributeError: module 'cv2' has no attribute 'TrackerCSRT_create'

In [3]:
# question 5

In [1]:
import numpy as np
import cv2

# Initialize video stream
video_path = "person_walking_in_bedroom.mp4"
cap = cv2.VideoCapture(video_path)

ret, frame = cap.read()
if not ret:
    print("Error: Cannot read video feed.")
    cap.release()
    exit()

# MANUALLY define bounding box coordinates: (x, y, width, height)
# Adjust these 4 values to fit your package location in the first frame
track_window = (200, 150, 100, 100)

x, y, w, h = track_window

# 1. Crop ROI and convert to HSV
roi = frame[y : y + h, x : x + w]
hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

# 2. Filter out low light/glare
mask = cv2.inRange(
    hsv_roi, np.array((0.0, 60.0, 32.0)), np.array((180.0, 255.0, 255.0))
)

# 3. Calculate 1D Hue histogram of target package
roi_hist = cv2.calcHist([hsv_roi], [0], mask, [180], [0, 180])
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

# Termination criteria: 10 iterations OR move < 1px
term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    dst = cv2.calcBackProject([hsv], [0], roi_hist, [0, 180], 1)

    # Apply Mean Shift
    ret, track_window = cv2.meanShift(dst, track_window, term_crit)

    # Draw result
    x, y, w, h = track_window
    cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # In Jupyter, use matplotlib or cv2_imshow instead of cv2.imshow
    cv2.imshow("Conveyor Belt Feed", frame)

    if cv2.waitKey(30) & 0xFF in (ord("q"), 27):
        break

cap.release()
cv2.destroyAllWindows()

In [2]:
# question 6

In [3]:
import numpy as np
import cv2

# Initialize video stream (0 for live camera or path to video file e.g., 'sports.mp4')
video_path = "person_walking_in_bedroom.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open video source.")
    exit()

# Read initial frame
ret, frame = cap.read()
if not ret or frame is None:
    print("Error: Cannot read initial frame.")
    cap.release()
    exit()

# --- TARGET SELECTION ---
# Option A: Interactive selection (Works in standard local Python scripts)
cv2.namedWindow("Select Player", cv2.WINDOW_NORMAL)
track_window = cv2.selectROI("Select Player", frame, fromCenter=False)
cv2.destroyWindow("Select Player")

x, y, w, h = track_window

# Fallback: Hardcode initial ROI (x, y, w, h) if running in Jupyter/headless environments
if w == 0 or h == 0:
    track_window = (200, 100, 80, 150)
    x, y, w, h = track_window

# 1. Crop player ROI and convert to HSV color space
roi = frame[y : y + h, x : x + w]
hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

# 2. Mask out non-informative pixels (extreme shadows / glare)
mask = cv2.inRange(
    hsv_roi, np.array((0.0, 60.0, 32.0)), np.array((180.0, 255.0, 255.0))
)

# 3. Calculate and normalize the 1D Hue histogram of the player target
roi_hist = cv2.calcHist([hsv_roi], [0], mask, [180], [0, 180])
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

# Termination criteria: Stop iteration when window moves < 1px or reaches 10 iterations
term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Convert frame to HSV and project color histogram model
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    dst = cv2.calcBackProject([hsv], [0], roi_hist, [0, 180], 1)

    # Apply CamShift to adapt window location, size, and orientation
    ret_camshift, track_window = cv2.CamShift(dst, track_window, term_crit)

    # CamShift returns a RotatedRect structure: ((center_x, center_y), (width, height), angle)
    pts = cv2.boxPoints(ret_camshift)
    pts = np.int32(pts)

    # Draw the adaptive, rotated tracking ellipse / bounding box around the player
    cv2.polylines(frame, [pts], True, (0, 255, 0), 2)

    # Extract orientation angle and bounding box dimensions
    (cx, cy), (box_w, box_h), angle = ret_camshift
    cv2.putText(
        frame,
        f"CamShift Tracking | Size: {int(box_w)}x{int(box_h)} | Angle: {int(angle)}deg",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 255, 255),
        2,
    )

    # Display video feed and back-projection heatmap
    cv2.imshow("CamShift Player Tracking", frame)
    cv2.imshow("Back-Projection Map", dst)

    key = cv2.waitKey(30) & 0xFF
    if key == ord("q") or key == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [4]:
# question 7

In [5]:
import numpy as np
import cv2

# Initialize video stream (0 for live camera or path to video file e.g., 'player_run.mp4')
video_path = "person_walking_in_bedroom.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open video feed.")
    exit()

# Parameters for Shi-Tomasi Corner Detection (Features to Track)
feature_params = dict(
    maxCorners=100, qualityLevel=0.3, minDistance=7, blockSize=7
)

# Parameters for Lucas-Kanade Optical Flow
lk_params = dict(
    winSize=(15, 15),
    maxLevel=2,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03),
)

# Read the initial frame
ret, old_frame = cap.read()
if not ret or old_frame is None:
    print("Error: Failed to read frame.")
    cap.release()
    exit()

old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

# Select initial feature points to track on the player
# In practice, pass an ROI mask over the player to track features only on them
p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

# Create a mask image for drawing optical flow tracking lines
mask = np.zeros_like(old_frame)


def get_direction_label(dx, dy, threshold=0.5):
    """Convert average displacement vector (dx, dy) into directional text."""
    magnitude = np.sqrt(dx**2 + dy**2)
    if magnitude < threshold:
        return "Stationary"

    # Calculate angle in degrees (-180 to 180)
    angle = np.arctan2(-dy, dx) * 180 / np.pi  # Invert dy due to image coords

    if -22.5 <= angle < 22.5:
        return "Right"
    elif 22.5 <= angle < 67.5:
        return "Up-Right"
    elif 67.5 <= angle < 112.5:
        return "Up"
    elif 112.5 <= angle < 157.5:
        return "Up-Left"
    elif angle >= 157.5 or angle < -157.5:
        return "Left"
    elif -157.5 <= angle < -112.5:
        return "Down-Left"
    elif -112.5 <= angle < -67.5:
        return "Down"
    elif -67.5 <= angle < -22.5:
        return "Down-Right"


while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    if p0 is not None and len(p0) > 0:
        # Calculate Lucas-Kanade Optical Flow: p1 = new points, st = status, err = error
        p1, st, err = cv2.calcOpticalFlowPyrLK(
            old_gray, frame_gray, p0, None, **lk_params
        )

        # Select good points (status == 1 means feature was found in new frame)
        if p1 is not None:
            good_new = p1[st == 1]
            good_old = p0[st == 1]

            displacements_x = []
            displacements_y = []

            # Draw tracking trajectories and compute displacement
            for i, (new, old) in enumerate(zip(good_new, good_old)):
                a, b = new.ravel()
                c, d = old.ravel()

                dx = a - c
                dy = b - d
                displacements_x.append(dx)
                displacements_y.append(dy)

                # Draw motion line and current point
                mask = cv2.line(
                    mask,
                    (int(a), int(b)),
                    (int(c), int(d)),
                    (0, 255, 0),
                    2,
                )
                frame = cv2.circle(
                    frame, (int(a), int(b)), 4, (0, 0, 255), -1
                )

            # Determine global average direction of all valid points
            if displacements_x:
                avg_dx = np.mean(displacements_x)
                avg_dy = np.mean(displacements_y)
                direction = get_direction_label(avg_dx, avg_dy)

                # Display direction and movement vector on output
                cv2.putText(
                    frame,
                    f"Movement Direction: {direction}",
                    (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 255, 255),
                    2,
                )
                cv2.putText(
                    frame,
                    f"Vector (dx, dy): ({avg_dx:.2f}, {avg_dy:.2f})",
                    (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (255, 255, 255),
                    1,
                )

            # Overlay trajectory mask on current frame
            output = cv2.add(frame, mask)

            # Update previous frame and previous points for next iteration
            old_gray = frame_gray.copy()
            p0 = good_new.reshape(-1, 1, 2)
        else:
            output = frame
    else:
        # Re-detect feature points if all existing points are lost
        p0 = cv2.goodFeaturesToTrack(frame_gray, mask=None, **feature_params)
        output = frame

    # Refresh feature points every 60 frames to keep tracking fresh
    if cv2.waitKey(30) & 0xFF == ord("r"):
        p0 = cv2.goodFeaturesToTrack(frame_gray, mask=None, **feature_params)
        mask = np.zeros_like(frame)

    cv2.imshow("Player Motion Direction (Lucas-Kanade)", output)

    key = cv2.waitKey(30) & 0xFF
    if key == ord("q") or key == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [6]:
# question 8

In [7]:
import cv2

# Initialize video feed (0 for live security camera, or path to video file e.g., 'entrance.mp4')
video_path = "person_walking_in_bedroom.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open entrance video feed.")
    exit()

# 1. Initialize HOG Person Detector
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

# List to hold active trackers for each detected person
trackers = []


def is_new_person(new_box, existing_trackers, iou_threshold=0.3):
    """Checks if a newly detected bounding box overlaps with an already tracked person."""
    nx, ny, nw, nh = new_box

    for tracker in existing_trackers:
        success, box = tracker.update(frame)
        if success:
            tx, ty, tw, th = [int(v) for v in box]

            # Compute Intersection over Union (IoU)
            x1 = max(nx, tx)
            y1 = max(ny, ty)
            x2 = min(nx + nw, tx + tw)
            y2 = min(ny + nh, ty + th)

            inter_area = max(0, x2 - x1) * max(0, y2 - y1)
            new_area = nw * nh
            tracked_area = tw * th
            union_area = new_area + tracked_area - inter_area

            if union_area > 0 and (inter_area / union_area) > iou_threshold:
                return False
    return True


frame_count = 0
# Re-detect every N frames to catch new people entering while tracking existing ones
REDETECT_INTERVAL = 15

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    # Resize frame for faster processing speed
    frame = cv2.resize(frame, (800, 600))
    active_trackers = []

    # --- STEP 1: Update Existing Trackers ---
    for tracker in trackers:
        success, bbox = tracker.update(frame)
        if success:
            active_trackers.append(tracker)
            x, y, w, h = [int(v) for v in bbox]

            # Draw bounding box and label for tracked person
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(
                frame,
                "Person",
                (x, y - 8),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 0),
                2,
            )

    trackers = active_trackers

    # --- STEP 2: Detect New Entrants (Periodically) ---
    if frame_count % REDETECT_INTERVAL == 0 or len(trackers) == 0:
        # Detect people in frame using HOG + SVM
        # winStride and padding balance detection speed and accuracy
        boxes, weights = hog.detectMultiScale(
            frame, winStride=(8, 8), padding=(8, 8), scale=1.05
        )

        for i, (x, y, w, h) in enumerate(boxes):
            # Confidence threshold to eliminate false positives
            if weights[i] > 0.6:
                new_box = (x, y, w, h)

                # Instantiate tracker if person is not already tracked
                if is_new_person(new_box, trackers):
                    new_tracker = cv2.TrackerCSRT_create()
                    new_tracker.init(frame, new_box)
                    trackers.append(new_tracker)

    # --- STEP 3: Display System Overlay ---
    cv2.putText(
        frame,
        f"Entrance Monitor | Active People: {len(trackers)}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2,
    )

    cv2.imshow("Building Security Entrance Feed", frame)

    key = cv2.waitKey(30) & 0xFF
    if key == ord("q") or key == 27:
        break

cap.release()
cv2.destroyAllWindows()

AttributeError: module 'cv2' has no attribute 'TrackerCSRT_create'

In [1]:
# question 9

In [3]:
import collections
import time
import cv2
import mediapipe as mp
import numpy as np

# Initialize MediaPipe Pose Model
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

# Initialize Video Feed (0 for live camera or path to file e.g., 'exercise.mp4')
video_path = "person_walking_in_bedroom.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open video feed.")
    exit()

# Buffer to store recent hip positions for velocity calculation (last 10 frames)
POSITION_BUFFER_SIZE = 10
hip_positions = collections.deque(maxlen=POSITION_BUFFER_SIZE)
prev_time = time.time()


def calculate_angle(a, b, c):
    """Calculates the 2D angle (in degrees) between three landmark joints."""
    a = np.array(a)  # First joint (e.g., Hip)
    b = np.array(b)  # Mid joint (e.g., Knee)
    c = np.array(c)  # End joint (e.g., Ankle)

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(
        a[1] - b[1], a[0] - b[0]
    )
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360 - angle

    return angle


def classify_activity(hip_speed, left_knee_angle, right_knee_angle):
    """Classifies human activity based on joint velocity and knee flexion angles."""
    # Thresholds (tuned for normalized screen coordinates per second)
    RUNNING_SPEED_THRESHOLD = 0.25
    WALKING_SPEED_THRESHOLD = 0.08

    # Average knee angle (lower angle = more bent leg)
    avg_knee_angle = (left_knee_angle + right_knee_angle) / 2.0

    if hip_speed > RUNNING_SPEED_THRESHOLD:
        return "Running", (0, 0, 255)  # Red
    elif hip_speed > WALKING_SPEED_THRESHOLD:
        return "Walking", (0, 255, 255)  # Yellow
    else:
        return "Standing", (0, 255, 0)  # Green


while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    curr_time = time.time()
    dt = curr_time - prev_time
    prev_time = curr_time

    # Convert BGR image to RGB for MediaPipe
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb_frame)

    activity = "Detecting..."
    color = (255, 255, 255)

    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark

        # Extract relevant landmark coordinates (Normalized x, y)
        left_hip = [
            landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x,
            landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y,
        ]
        right_hip = [
            landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x,
            landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y,
        ]

        left_knee = [
            landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x,
            landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y,
        ]
        right_knee = [
            landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x,
            landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y,
        ]

        left_ankle = [
            landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x,
            landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y,
        ]
        right_ankle = [
            landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x,
            landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y,
        ]

        # Calculate midpoint of hips to track core body position
        mid_hip = [(left_hip[0] + right_hip[0]) / 2, (left_hip[1] + right_hip[1]) / 2]
        hip_positions.append((mid_hip, curr_time))

        # Calculate Hip Velocity (Distance moved / time elapsed)
        hip_speed = 0.0
        if len(hip_positions) >= 2:
            (pos_old, time_old) = hip_positions[0]
            (pos_new, time_new) = hip_positions[-1]
            time_diff = time_new - time_old

            if time_diff > 0:
                dist = np.sqrt(
                    (pos_new[0] - pos_old[0]) ** 2
                    + (pos_new[1] - pos_old[1]) ** 2
                )
                hip_speed = dist / time_diff

        # Calculate joint angles
        left_knee_angle = calculate_angle(left_hip, left_knee, left_ankle)
        right_knee_angle = calculate_angle(right_hip, right_knee, right_ankle)

        # Classify current frame's activity
        activity, color = classify_activity(
            hip_speed, left_knee_angle, right_knee_angle
        )

        # Draw Pose Skeleton overlay
        mp_drawing.draw_landmarks(
            frame,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(
                color=(245, 117, 66), thickness=2, circle_radius=2
            ),
            mp_drawing.DrawingSpec(
                color=(245, 66, 230), thickness=2, circle_radius=2
            ),
        )

    # UI Overlay
    cv2.rectangle(frame, (0, 0), (350, 60), (0, 0, 0), -1)
    cv2.putText(
        frame,
        f"Activity: {activity}",
        (10, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        color,
        2,
        cv2.LINE_AA,
    )

    cv2.imshow("Human Activity Recognition", frame)

    key = cv2.waitKey(30) & 0xFF
    if key == ord("q") or key == 27:
        break

cap.release()
cv2.destroyAllWindows()

ModuleNotFoundError: No module named 'mediapipe'

In [2]:
# question 10

In [4]:
from collections import deque
import cv2
import numpy as np

# Initialize video feed (0 for live CCTV feed, or path to video file e.g., 'lab_cctv.mp4')
video_path = "person_walking_in_bedroom.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open CCTV video feed.")
    exit()

# Parameters for Farneback Dense Optical Flow
farneback_params = dict(
    pyr_scale=0.5,
    levels=3,
    winsize=15,
    iterations=3,
    poly_n=5,
    poly_sigma=1.2,
    flags=0,
)

# Rolling history buffer for dynamic baseline statistical calculation
HISTORY_WINDOW = 100  # Number of frames to compute running mean/std
motion_magnitude_history = deque(maxlen=HISTORY_WINDOW)

# Anomaly threshold: Number of standard deviations above mean to trigger alert
Z_SCORE_THRESHOLD = 3.0
MIN_BASELINE_FRAMES = 30  # Minimum frames required to establish baseline

# Read initial frame
ret, prev_frame = cap.read()
if not ret or prev_frame is None:
    print("Error: Failed to read video feed.")
    cap.release()
    exit()

prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 1. Compute Dense Optical Flow using Farneback algorithm
    flow = cv2.calcOpticalFlowFarneback(
        prev_gray, curr_gray, None, **farneback_params
    )

    # 2. Calculate magnitude and angle of motion vectors for all pixels
    magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])

    # Filter out minor background sensor noise
    motion_mask = magnitude > 1.5
    active_motion_magnitude = magnitude[motion_mask]

    # Calculate average motion intensity in current frame
    if len(active_motion_magnitude) > 0:
        current_mean_motion = np.mean(active_motion_magnitude)
    else:
        current_mean_motion = 0.0

    # 3. Anomaly Detection Logic via Moving Z-Score
    is_anomaly = False
    z_score = 0.0

    if len(motion_magnitude_history) >= MIN_BASELINE_FRAMES:
        baseline_mean = np.mean(motion_magnitude_history)
        baseline_std = np.std(motion_magnitude_history) + 1e-5  # Prevent div-by-zero

        # Z-Score: measures how many standard deviations the current frame deviates from baseline
        z_score = (current_mean_motion - baseline_mean) / baseline_std

        if z_score > Z_SCORE_THRESHOLD:
            is_anomaly = True

    # Update dynamic baseline history
    motion_magnitude_history.append(current_mean_motion)

    # 4. Generate Visualization Overlay
    # Map motion vectors to HSV color wheel (Hue = direction, Value = speed)
    hsv = np.zeros_like(frame)
    hsv[..., 1] = 255
    hsv[..., 0] = angle * 180 / np.pi / 2
    hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)
    flow_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    # Draw Status and Alert Overlay
    if is_anomaly:
        # Red Border Alert for unusual movement
        cv2.rectangle(
            frame, (0, 0), (frame.shape[1], frame.shape[0]), (0, 0, 255), 10
        )
        cv2.rectangle(frame, (0, 0), (450, 60), (0, 0, 255), -1)
        cv2.putText(
            frame,
            "ALERT: UNUSUAL MOVEMENT DETECTED!",
            (10, 38),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )
    else:
        # Green status bar for normal movement
        cv2.rectangle(frame, (0, 0), (320, 50), (0, 0, 0), -1)
        cv2.putText(
            frame,
            "Status: Normal Activity",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2,
            cv2.LINE_AA,
        )

    # Display Z-Score and Flow metrics
    cv2.putText(
        frame,
        f"Motion Index: {current_mean_motion:.2f} | Z-Score: {z_score:.2f}",
        (10, frame.shape[0] - 15),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 255, 255),
        1,
        cv2.LINE_AA,
    )

    # Render display windows
    cv2.imshow("Restricted Lab CCTV Monitor", frame)
    cv2.imshow("Optical Flow Heatmap", flow_rgb)

    prev_gray = curr_gray.copy()

    key = cv2.waitKey(30) & 0xFF
    if key == ord("q") or key == 27:
        break

cap.release()
cv2.destroyAllWindows()